# TCN-UNet — Subject-Dependent Split (scEEG -> iEEG)

**Split:** for EVERY subject individually, a single stratified **70% train / 10% validation /
20% test** split. A **separate** TCN-UNet is trained per subject. Leftover non-IED segments
belonging to that subject are added to its TEST portion only.



### A different model family: TCN-UNet (Temporal Convolutional Network)

Per your reference document, this swaps the mapping model for a **TCN-UNet**: dilated
residual convolutional blocks in a U-Net-style encoder/decoder, with skip connections. No
VAE, no GAN/discriminator, no attention -- a more classical, purely convolutional-temporal
architecture.

**Adapted from the document for this dataset's fixed 250 ms (64-sample) segments:**
- The document's suggested receptive field (~1-2 seconds via 8 dilated layers) assumes a
  much longer window than the paper's fixed 64-sample segment length -- can't change that
  (fixed by the dataset). Instead, the dilation schedule (1, 2, 4 at full resolution) is
  sized so the receptive field already covers the *entire* 64-sample window, which is the
  most context available regardless.
- **Non-causal** (bidirectional/symmetric) dilated convolutions, as recommended for offline
  (non-real-time) mapping -- every output position sees both past and future context within
  the segment.
- **Encoder-decoder ("TCN-UNet")** with skip connections at matching resolutions, exactly as
  suggested, to help preserve spike sharpness that a plain stacked TCN can smooth over.
- **Output layer**: plain 1x1 conv, no activation -- iEEG amplitude is continuous and
  unbounded, so (unlike the tanh-bounded generators used before) there's no need to clip the
  target into [-1, 1]. Both scEEG and iEEG now use plain per-channel z-score normalization.
- **Two-stage training**, as specified: Stage 1 pretrains on all data with uniform weighting
  (plain reconstruction loss); Stage 2 fine-tunes with IED-oversampled/upweighted batches, so
  the model prioritizes spike morphology precisely -- this is what's meant to push IED
  PCORR/COSSIM above non-IED's.
- **Loss = L1 + (1-Pearson) + (1-Cosine) + Spectral + IED-weighting**, matching the document's
  formula, with Pearson/Cosine as *direct* differentiable loss terms (not just L1) since
  they're literally the metrics being evaluated -- optimizing them directly closes the gap
  between the training objective and the reported metric.
- **Margin loss** (new, per the document): `max(0, target_margin - (corr_IED - corr_nonIED))`
  -- an explicit penalty whenever the IED/non-IED correlation gap falls short of
  `MARGIN_TARGET`, directly enforcing IED > non-IED rather than just hoping upweighting
  produces it as a side effect.

**Everything else is identical to the previous (cross-attention) version** -- same dataset
file, same preprocessing, same no-leakage grouping, same leftover-non-IED-in-test-only
handling, same literal (non-z-scored) MSE/PCORR/COSSIM metrics, same shuffle-control
diagnostic, same epochs+patience-style training budget (extended here to the two stages the
document specifically calls for).


## 1. Setup & config

In [ ]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import os, copy

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

DATA_PATH = "balanced_segmented_dataset.npz"
LEFTOVER_PATH = "leftover_non_ied_segments.npz"   # extra non-IED segments -- TRAIN split only

IED_LOSS_WEIGHT = 2.5   # IED segments count this many times as much in the training loss as
                        # non-IED -- makes the optimizer prioritize IED fidelity, targeting
                        # IED PCORR/COSSIM > non-IED (see Section 5 for why this is needed)

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.10, 0.20

IED_LOSS_WEIGHT = 3.0       # Stage 2 oversampling/loss weight for IED, per the document (3-5x)
MARGIN_TARGET = 0.15        # Stage 2 margin loss target: corr_IED - corr_nonIED >= this

# ---- model size ----
BASE_CH = 32

# ---- two-stage training ----
STAGE1_EPOCHS = 40          # pretrain, uniform weighting, fixed length
STAGE2_EPOCHS, PATIENCE = 100, 20   # fine-tune, IED-weighted + margin loss, early-stopped
LR = 1e-3
LAM1, LAM2, LAM3, LAM4, LAM_MARGIN = 1.0, 2.0, 1.0, 0.5, 2.0   # L1, Pearson, Cosine, Spectral, margin
BATCH_SIZE = 32


## 2. Load the balanced segmented dataset (+ leftover non-IED segments)

In [ ]:

data = np.load(DATA_PATH, allow_pickle=True)

X_eeg  = data["X_eeg"].astype(np.float32)
X_ieeg = data["X_ieeg"].astype(np.float32)
y      = data["y"].astype(np.int64)
subject_ids = data["subject_ids"]
eeg_names = list(data["eeg_names"])
fo_names  = list(data["fo_names"])
fs = float(data["fs"])
L  = X_eeg.shape[1]
M  = X_eeg.shape[2]
Mb = X_ieeg.shape[2]

unique_subjects = sorted(np.unique(subject_ids).tolist())
print(f"Total segments: {len(y)}  |  scEEG shape: {X_eeg.shape}  |  iEEG shape: {X_ieeg.shape}")
print(f"Subjects ({len(unique_subjects)}):", unique_subjects)
print(f"IED: {int((y==1).sum())}   Non-IED: {int((y==0).sum())}")

# ---- leftover non-IED segments (loaded now, merged into TRAIN only after the split below) ----
if os.path.exists(LEFTOVER_PATH):
    leftover_raw = np.load(LEFTOVER_PATH, allow_pickle=True)
    lf_keys = list(leftover_raw.keys())
    print(f"\nFound {LEFTOVER_PATH}, keys: {lf_keys}")
    lf_X_eeg = leftover_raw["X_eeg"].astype(np.float32) if "X_eeg" in lf_keys else leftover_raw["eeg"].astype(np.float32)
    lf_X_ieeg = leftover_raw["X_ieeg"].astype(np.float32) if "X_ieeg" in lf_keys else leftover_raw["ieeg"].astype(np.float32)
    assert lf_X_eeg.shape[1:] == (L, M), f"leftover scEEG shape {lf_X_eeg.shape} doesn't match main dataset (*, {L}, {M})"
    assert lf_X_ieeg.shape[1:] == (L, Mb), f"leftover iEEG shape {lf_X_ieeg.shape} doesn't match main dataset (*, {L}, {Mb})"
    lf_y = leftover_raw["y"].astype(np.int64) if "y" in lf_keys else np.zeros(len(lf_X_eeg), dtype=np.int64)
    assert (lf_y == 0).all(), "leftover_non_ied_segments.npz contains a non-zero label -- expected all non-IED (y=0)"
    lf_subject_ids = leftover_raw["subject_ids"] if "subject_ids" in lf_keys else None
    if lf_subject_ids is None:
        print("WARNING: leftover file has no 'subject_ids' -- cannot route it per-subject or "
              "verify it avoids leaking into a held-out subject/group. It will still be added "
              "to training pools where subject identity doesn't matter, and SKIPPED wherever "
              "subject-safety can't be guaranteed.")
    print(f"Leftover non-IED segments available: {len(lf_X_eeg)}")
else:
    lf_X_eeg = lf_X_ieeg = lf_y = lf_subject_ids = None
    print(f"\n{LEFTOVER_PATH} not found -- proceeding without extra non-IED segments.")


## 3. Segment-level preprocessing

In [ ]:

# ============================================================================
# Segment-level preprocessing.
#
# Kept from before (validated safe by direct testing):
#   - 50 Hz notch via spectral bin removal (stable on short windows, unlike
#     filtfilt whose settling time exceeds a 250 ms window)
#   - Common average reference (CAR), scEEG only, per the paper
#   - NOT full linear detrend -- a controlled test showed detrending a 250 ms
#     window removes real shared slow-wave structure, hurting correlation.
#
# CHANGED: baseline correction (subtract mean of first few samples) is replaced
# with full per-segment DC removal (subtract the WHOLE segment's own mean).
# This is safe -- Pearson correlation is provably invariant to a constant
# shift, verified directly (identical correlation with/without mean removal)
# -- and it's what makes literal COSSIM converge toward PCORR: cosine
# similarity computed on zero-mean data is mathematically equal to Pearson
# correlation on the original data.
# ============================================================================

APPLY_NOTCH = True
APPLY_CAR = True                 # scEEG only, per the paper
APPLY_MEAN_REMOVAL = True        # full per-segment DC removal (safe -- see above)
APPLY_DETREND = False            # full linear detrend -- OFF, shown to hurt correlation
NOTCH_HZ = 50.0
NOTCH_BW = 4.0

def notch_fft(x, fs, freq=NOTCH_HZ, bw=NOTCH_BW):
    '''x: (..., L) with time as the LAST axis.'''
    L_ = x.shape[-1]
    freqs = np.fft.rfftfreq(L_, d=1.0 / fs)
    mask = (freqs >= freq - bw / 2) & (freqs <= freq + bw / 2)
    Xf = np.fft.rfft(x, axis=-1)
    Xf[..., mask] = 0
    return np.fft.irfft(Xf, n=L_, axis=-1)

def preprocess_segments(X, fs, apply_car=False):
    '''X: (N, L, C) -> (N, L, C).'''
    from scipy.signal import detrend as _scipy_detrend
    Xp = X.copy()
    if APPLY_DETREND:
        Xp = _scipy_detrend(Xp, axis=1, type='linear')
    elif APPLY_MEAN_REMOVAL:
        Xp = Xp - Xp.mean(axis=1, keepdims=True)
    if APPLY_NOTCH:
        Xt = np.moveaxis(Xp, 1, -1)
        Xt = notch_fft(Xt, fs)
        Xp = np.moveaxis(Xt, -1, 1)
    if apply_car:
        Xp = Xp - Xp.mean(axis=2, keepdims=True)
    return Xp.astype(np.float32)

print(f"Preprocessing scEEG (CAR={APPLY_CAR}, notch={APPLY_NOTCH}, mean_removal={APPLY_MEAN_REMOVAL})...")
X_eeg  = preprocess_segments(X_eeg,  fs, apply_car=APPLY_CAR)
print(f"Preprocessing iEEG (CAR=False, notch={APPLY_NOTCH}, mean_removal={APPLY_MEAN_REMOVAL})...")
X_ieeg = preprocess_segments(X_ieeg, fs, apply_car=False)
assert not np.isnan(X_eeg).any() and not np.isnan(X_ieeg).any(), "NaNs introduced by preprocessing!"

if lf_X_eeg is not None:
    print("Preprocessing leftover non-IED segments with the SAME functions/flags...")
    lf_X_eeg  = preprocess_segments(lf_X_eeg,  fs, apply_car=APPLY_CAR)
    lf_X_ieeg = preprocess_segments(lf_X_ieeg, fs, apply_car=False)

print("Done. X_eeg / X_ieeg (and leftover, if present) are now preprocessed.")


## 4. Model architecture: TCN-UNet

In [ ]:

# ============================================================================
# TCN-UNet: dilated residual TCN blocks in a U-Net encoder/decoder with skip
# connections. Non-causal (symmetric padding). Output: linear, unbounded.
# ============================================================================

class ResidualTCNBlock(nn.Module):
    '''Dilated Conv1D -> WeightNorm -> ReLU -> Dropout, x2, + residual
    (1x1 conv if channel dims differ). Symmetric ("same") padding -- non-causal,
    since this is offline (not real-time) mapping.'''
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.utils.parametrizations.weight_norm(nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation))
        self.conv2 = nn.utils.parametrizations.weight_norm(nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, dilation=dilation))
        self.drop = nn.Dropout(dropout)
        self.act = nn.ReLU()
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        h = self.drop(self.act(self.conv1(x)))
        h = self.drop(self.act(self.conv2(h)))
        return h + self.skip(x)


class TCNStack(nn.Module):
    def __init__(self, in_ch, out_ch, dilations, dropout=0.1):
        super().__init__()
        blocks = []
        c_in = in_ch
        for d in dilations:
            blocks.append(ResidualTCNBlock(c_in, out_ch, dilation=d, dropout=dropout))
            c_in = out_ch
        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        return self.blocks(x)


class TCNUNet(nn.Module):
    '''Encoder: TCNStack + strided-conv downsample, x2 levels. Bottleneck: TCNStack.
    Decoder: transposed-conv upsample + skip-concat + TCNStack, mirrored. Output:
    1x1 conv, no activation.'''
    def __init__(self, in_ch=20, out_ch=12, L=64, base_ch=64, dropout=0.1):
        super().__init__()
        self.enc0 = TCNStack(in_ch, base_ch, dilations=[1, 2, 4], dropout=dropout)
        self.down0 = nn.Conv1d(base_ch, base_ch * 2, kernel_size=4, stride=2, padding=1)
        self.enc1 = TCNStack(base_ch * 2, base_ch * 2, dilations=[1, 2], dropout=dropout)
        self.down1 = nn.Conv1d(base_ch * 2, base_ch * 4, kernel_size=4, stride=2, padding=1)
        self.bottleneck = TCNStack(base_ch * 4, base_ch * 4, dilations=[1, 2], dropout=dropout)
        self.up1 = nn.ConvTranspose1d(base_ch * 4, base_ch * 2, kernel_size=4, stride=2, padding=1)
        self.dec1 = TCNStack(base_ch * 2 + base_ch * 2, base_ch * 2, dilations=[1, 2], dropout=dropout)
        self.up0 = nn.ConvTranspose1d(base_ch * 2, base_ch, kernel_size=4, stride=2, padding=1)
        self.dec0 = TCNStack(base_ch + base_ch, base_ch, dilations=[1, 2], dropout=dropout)
        self.out_conv = nn.Conv1d(base_ch, out_ch, kernel_size=1)

    def forward(self, X):   # X: (B, L, M)
        x = X.permute(0, 2, 1)
        s0 = self.enc0(x)
        x = self.down0(s0)
        s1 = self.enc1(x)
        x = self.down1(s1)
        x = self.bottleneck(x)
        x = self.up1(x)
        x = torch.cat([x, s1], dim=1)
        x = self.dec1(x)
        x = self.up0(x)
        x = torch.cat([x, s0], dim=1)
        x = self.dec0(x)
        y = self.out_conv(x)
        return y.permute(0, 2, 1)   # (B, L, out_ch)


## 5. Loss functions

In [ ]:

# ============================================================================
# Loss = lam1*L1 + lam2*(1-Pearson) + lam3*(1-Cosine) + lam4*Spectral + IED-weighting,
# plus a margin loss enforcing corr_IED - corr_nonIED >= MARGIN_TARGET. All terms
# support a per-sample weight (used for IED upweighting in Stage 2).
# ============================================================================

def loss_l1(y_real, y_est, w=None):
    per = torch.abs(y_real - y_est).mean(dim=(1, 2))
    return (per * w).sum() / w.sum() if w is not None else per.mean()

def _pearson_per_sample(y_real, y_est, eps=1e-8):
    yr = y_real - y_real.mean(dim=1, keepdim=True)
    ye = y_est - y_est.mean(dim=1, keepdim=True)
    num = (yr * ye).sum(dim=1)
    den = torch.sqrt((yr ** 2).sum(dim=1) * (ye ** 2).sum(dim=1) + eps)
    return (num / (den + eps)).mean(dim=1)   # (B,) raw correlation (not 1-corr)

def loss_pearson(y_real, y_est, w=None):
    per = 1 - _pearson_per_sample(y_real, y_est)
    return (per * w).sum() / w.sum() if w is not None else per.mean()

def zscore_time(x, eps=1e-6):
    mu = x.mean(dim=1, keepdim=True)
    sd = x.std(dim=1, keepdim=True)
    return (x - mu) / (sd + eps)

def loss_cosine(y_real, y_est, w=None, eps=1e-8):
    yr, ye = zscore_time(y_real, eps), zscore_time(y_est, eps)
    num = (yr * ye).sum(dim=1)
    den = torch.norm(yr, dim=1) * torch.norm(ye, dim=1)
    per = (1 - num / (den + eps)).mean(dim=1)
    return (per * w).sum() / w.sum() if w is not None else per.mean()

def loss_spectral(y_real, y_est, w=None):
    Yr = torch.fft.rfft(y_real, dim=1).abs()
    Ye = torch.fft.rfft(y_est, dim=1).abs()
    per = torch.abs(Yr - Ye).mean(dim=(1, 2))
    return (per * w).sum() / w.sum() if w is not None else per.mean()

def margin_loss(y_real, y_est, labels, target_margin=0.2):
    '''max(0, target_margin - (mean_corr_IED - mean_corr_nonIED)) -- explicitly
    enforces the IED/non-IED correlation gap, per the document.'''
    per_corr = _pearson_per_sample(y_real, y_est)
    ied_mask, non_mask = labels > 0.5, labels <= 0.5
    if ied_mask.sum() == 0 or non_mask.sum() == 0:
        return torch.tensor(0.0, device=y_real.device)
    gap = per_corr[ied_mask].mean() - per_corr[non_mask].mean()
    return torch.clamp(target_margin - gap, min=0.0)

def loss_total(y_real, y_est, labels, lam1=1.0, lam2=2.0, lam3=1.0, lam4=0.5, lam_margin=2.0,
               target_margin=0.2, sample_w=None):
    ll1 = loss_l1(y_real, y_est, sample_w)
    lpc = loss_pearson(y_real, y_est, sample_w)
    lcos = loss_cosine(y_real, y_est, sample_w)
    lspec = loss_spectral(y_real, y_est, sample_w)
    lmargin = margin_loss(y_real, y_est, labels, target_margin)
    total = lam1*ll1 + lam2*lpc + lam3*lcos + lam4*lspec + lam_margin*lmargin
    return total, ll1, lpc, lcos, lspec, lmargin

def ied_upweight(labels, ied_weight=IED_LOSS_WEIGHT):
    return torch.where(labels > 0.5, torch.full_like(labels, ied_weight), torch.ones_like(labels))


In [ ]:

class SegSet(Dataset):
    def __init__(self, sc, ie, lab):
        self.sc  = torch.tensor(sc,  dtype=torch.float32)
        self.ie  = torch.tensor(ie,  dtype=torch.float32)
        self.lab = torch.tensor(lab, dtype=torch.float32)
    def __len__(self): return len(self.lab)
    def __getitem__(self, i): return self.sc[i], self.ie[i], self.lab[i]

def make_balanced_sampler(labels):
    '''WeightedRandomSampler that gives IED and non-IED roughly equal representation
    per batch, regardless of how imbalanced the underlying training set is (e.g.
    after adding the leftover non-IED segments).'''
    labels = np.asarray(labels)
    class_counts = np.array([max(1, (labels == c).sum()) for c in (0, 1)])
    class_weight = 1.0 / class_counts
    sample_weights = class_weight[labels.astype(int)]
    return WeightedRandomSampler(weights=torch.tensor(sample_weights, dtype=torch.float32),
                                  num_samples=len(labels), replacement=True)


In [ ]:

# ============================================================================
# Per-channel z-score normalization, fit on TRAIN data only, for BOTH scEEG and
# iEEG. Unlike the tanh-bounded generators used before, the TCN-UNet's output
# layer is linear/unbounded (per the document -- iEEG amplitude is continuous),
# so there's no need to percentile-clip the target into [-1, 1] anymore.
# ============================================================================

def fit_scaler(X_train):
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std  = X_train.std(axis=(0, 1), keepdims=True) + 1e-10
    return mean, std

def apply_scaler(X, mean, std):
    return (X - mean) / std

# kept as aliases so the rest of the notebook (and the shuffle-control snippet) can use the
# same fit_scaler_sc/apply_scaler_sc/fit_scaler_ie/apply_scaler_ie names as before
def fit_scaler_sc(X_train): return fit_scaler(X_train)
def apply_scaler_sc(X, mean, std): return apply_scaler(X, mean, std)
def fit_scaler_ie(X_train):
    mean, std = fit_scaler(X_train)
    return (mean, std)
def apply_scaler_ie(X, scale):
    mean, std = scale
    return apply_scaler(X, mean, std)


## 6. Metrics: literal MSE / PCORR / COSSIM (paper Eqs. 14-16), for IED, Non-IED, and Combined

In [ ]:

def score_mapping(model, loader, device=DEVICE):
    '''Literal MSE / PCORR / COSSIM (paper Eqs. 14-16), no re-standardization.'''
    model.eval()
    mse_vals, pcorr_vals, cos_vals, label_vals = [], [], [], []
    with torch.no_grad():
        for sc, ie, lab in loader:
            sc, ie = sc.to(device), ie.to(device)
            y_est = model(sc)
            ie_np, ye_np = ie.cpu().numpy(), y_est.cpu().numpy()
            for i in range(ie_np.shape[0]):
                for j in range(ie_np.shape[2]):
                    yv, yev = ie_np[i, :, j], ye_np[i, :, j]
                    mse_vals.append(np.mean((yv - yev) ** 2))
                    pcorr_vals.append(pearsonr(yv, yev)[0])
                    denom = np.linalg.norm(yv) * np.linalg.norm(yev)
                    cos_vals.append(float(np.dot(yv, yev) / (denom + 1e-8)))
                    label_vals.append(lab[i].item())
    mse_vals, pcorr_vals, cos_vals, label_vals = (np.array(a) for a in
        (mse_vals, pcorr_vals, cos_vals, label_vals))

    def summarize(mask):
        if mask.sum() == 0:
            return dict(MSE=np.nan, PCORR=np.nan, COSSIM=np.nan)
        return dict(MSE=float(np.mean(mse_vals[mask])),
                    PCORR=float(np.mean(pcorr_vals[mask])),
                    COSSIM=float(np.mean(cos_vals[mask])))

    return {
        "Combined": summarize(np.ones_like(label_vals, dtype=bool)),
        "IED":      summarize(label_vals == 1),
        "Non-IED":  summarize(label_vals == 0),
    }


## 7. Training loop (two-stage)

In [ ]:

def fit_model(model, train_loader_uniform, train_loader_weighted, val_loader, device=DEVICE,
              stage1_epochs=30, stage2_epochs=100, patience=20, lr=1e-3,
              lam1=1.0, lam2=2.0, lam3=1.0, lam4=0.5, lam_margin=2.0, target_margin=0.2, verbose=True):
    '''Two-stage training, per the document:
      Stage 1 (stage1_epochs, fixed length, no early stopping): pretrain on ALL data with
        UNIFORM weighting -- plain reconstruction loss, learns the general scEEG<->iEEG mapping.
      Stage 2 (up to stage2_epochs, early-stopped via `patience` on validation PCORR):
        fine-tune with IED-oversampled/upweighted batches + the margin loss, so the model
        prioritizes spike morphology and IED ends up with higher PCORR/COSSIM than non-IED.
    AdamW + ReduceLROnPlateau scheduled on validation PCORR directly (the actual target
    metric), not just loss.'''
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)

    if verbose: print(f"--- Stage 1: pretrain ({stage1_epochs} epochs, uniform weighting) ---")
    for epoch in range(stage1_epochs):
        model.train()
        for sc, ie, lab in train_loader_uniform:
            sc, ie = sc.to(device), ie.to(device)
            y_est = model(sc)
            loss, *_ = loss_total(ie, y_est, torch.zeros(len(ie), device=device),  # margin loss off in stage 1
                                   lam1, lam2, lam3, lam4, lam_margin=0.0, target_margin=target_margin)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        if verbose and (epoch % 10 == 0 or epoch == stage1_epochs - 1):
            model.eval()
            with torch.no_grad():
                va_corr = np.mean([( 1 - loss_pearson(ie.to(device), model(sc.to(device))) ).item()
                                    for sc, ie, _ in val_loader])
            print(f"  stage1 epoch {epoch:3d} | train_loss {loss.item():.3f} | val_PCORR {va_corr:.3f}")

    if verbose: print(f"--- Stage 2: fine-tune (up to {stage2_epochs} epochs, IED-weighted + margin loss) ---")
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=max(3, patience // 2))
    best_score, wait, best_state = -float('inf'), 0, None

    for epoch in range(stage2_epochs):
        model.train()
        for sc, ie, lab in train_loader_weighted:
            sc, ie, lab = sc.to(device), ie.to(device), lab.to(device)
            sample_w = ied_upweight(lab)
            y_est = model(sc)
            loss, *_ = loss_total(ie, y_est, lab, lam1, lam2, lam3, lam4, lam_margin, target_margin, sample_w)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()

        model.eval()
        va_corr = va_mse = 0.0
        with torch.no_grad():
            for sc, ie, lab in val_loader:
                sc, ie = sc.to(device), ie.to(device)
                y_est = model(sc)
                va_corr += (1 - loss_pearson(ie, y_est)).item()
                va_mse += torch.mean((ie - y_est) ** 2).item()
        va_corr /= len(val_loader); va_mse /= len(val_loader)
        sched.step(va_corr)

        if va_corr > best_score:
            best_score, wait = va_corr, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1

        if verbose and (epoch % 10 == 0 or epoch == stage2_epochs - 1):
            print(f"  stage2 epoch {epoch:3d} | val_PCORR {va_corr:.3f} val_MSE {va_mse:.3f}  (wait {wait}/{patience})")

        if wait >= patience:
            if verbose:
                print(f"  early stopping at stage2 epoch {epoch} (no improvement for {patience} epochs)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


In [ ]:

def metrics_dict_to_row(d):
    row = {}
    for cls in ["Combined", "IED", "Non-IED"]:
        for met in ["MSE", "PCORR", "COSSIM"]:
            row[(met, cls)] = d[cls][met]
    return row

def build_results_table(rows_dict, index_name="Subject"):
    flat_rows = {label: metrics_dict_to_row(d) for label, d in rows_dict.items()}
    df = pd.DataFrame.from_dict(flat_rows, orient="index")
    df.columns = pd.MultiIndex.from_tuples(df.columns, names=["Metric", "Class"])
    df.index.name = index_name
    if len(df) > 1:
        df.loc["Mean"] = df.mean(numeric_only=True)
    return df.round(3)


## 8. Per-subject 70/10/20 split, scaling, and leftover merge

One stratified split per subject. Scalers (Section 5) fit on that subject's TRAIN portion
only. Leftover non-IED segments for that subject (if any) are appended to TEST only. Builds
TWO training loaders per subject: a uniform one (Stage 1) and a class-balanced one (Stage 2).

In [ ]:

from sklearn.model_selection import train_test_split

def make_subject_split(sc, ie, lab, test_size=TEST_FRAC, val_size=VAL_FRAC, seed=SEED):
    Xtr_sc, Xte_sc, Xtr_ie, Xte_ie, ytr, yte = train_test_split(
        sc, ie, lab, test_size=test_size, random_state=seed, stratify=lab)
    val_ratio = val_size / (1 - test_size)
    Xtr_sc, Xva_sc, Xtr_ie, Xva_ie, ytr, yva = train_test_split(
        Xtr_sc, Xtr_ie, ytr, test_size=val_ratio, random_state=seed, stratify=ytr)
    return Xtr_sc, Xva_sc, Xte_sc, Xtr_ie, Xva_ie, Xte_ie, ytr, yva, yte

subject_data = {}
for subj in unique_subjects:
    mask = subject_ids == subj
    Xtr_sc, Xva_sc, Xte_sc, Xtr_ie, Xva_ie, Xte_ie, ytr, yva, yte = make_subject_split(
        X_eeg[mask], X_ieeg[mask], y[mask])

    sc_mean, sc_std = fit_scaler_sc(Xtr_sc)
    ie_scale = fit_scaler_ie(Xtr_ie)
    Xtr_sc_n, Xva_sc_n, Xte_sc_n = (apply_scaler_sc(x, sc_mean, sc_std) for x in (Xtr_sc, Xva_sc, Xte_sc))
    Xtr_ie_n, Xva_ie_n, Xte_ie_n = (apply_scaler_ie(x, ie_scale) for x in (Xtr_ie, Xva_ie, Xte_ie))

    if lf_X_eeg is not None and lf_subject_ids is not None:
        lf_mask = lf_subject_ids == subj
        if lf_mask.sum() > 0:
            lf_sc_n = apply_scaler_sc(lf_X_eeg[lf_mask], sc_mean, sc_std)
            lf_ie_n = apply_scaler_ie(lf_X_ieeg[lf_mask], ie_scale)
            Xte_sc_n = np.concatenate([Xte_sc_n, lf_sc_n], axis=0)
            Xte_ie_n = np.concatenate([Xte_ie_n, lf_ie_n], axis=0)
            yte = np.concatenate([yte, np.zeros(lf_mask.sum(), dtype=np.int64)], axis=0)

    subject_data[subj] = dict(Xtr_sc=Xtr_sc_n, Xva_sc=Xva_sc_n, Xte_sc=Xte_sc_n,
                               Xtr_ie=Xtr_ie_n, Xva_ie=Xva_ie_n, Xte_ie=Xte_ie_n,
                               ytr=ytr, yva=yva, yte=yte)
    print(f"{subj}: train={len(ytr)} (IED={int(ytr.sum())}, non-IED={int((ytr==0).sum())})  "
          f"val={len(yva)}  test={len(yte)} (IED={int(yte.sum())}, non-IED={int((yte==0).sum())})")


## 9. Train one TCN-UNet per subject (two-stage)

In [ ]:

per_subject_metrics = {}
trained_models = {}

for subj in unique_subjects:
    print(f"\n=== {subj} ===")
    d = subject_data[subj]

    tr_uniform = DataLoader(SegSet(d["Xtr_sc"], d["Xtr_ie"], d["ytr"]), batch_size=BATCH_SIZE, shuffle=True)
    sampler = make_balanced_sampler(d["ytr"])
    tr_weighted = DataLoader(SegSet(d["Xtr_sc"], d["Xtr_ie"], d["ytr"]), batch_size=BATCH_SIZE, sampler=sampler)
    va_loader = DataLoader(SegSet(d["Xva_sc"], d["Xva_ie"], d["yva"]), batch_size=16, shuffle=False)
    te_loader = DataLoader(SegSet(d["Xte_sc"], d["Xte_ie"], d["yte"]), batch_size=16, shuffle=False)

    model = TCNUNet(in_ch=M, out_ch=Mb, L=L, base_ch=BASE_CH)
    model = fit_model(model, tr_uniform, tr_weighted, va_loader, device=DEVICE,
                       stage1_epochs=STAGE1_EPOCHS, stage2_epochs=STAGE2_EPOCHS, patience=PATIENCE, lr=LR,
                       lam1=LAM1, lam2=LAM2, lam3=LAM3, lam4=LAM4, lam_margin=LAM_MARGIN,
                       target_margin=MARGIN_TARGET, verbose=False)

    metrics = score_mapping(model, te_loader, device=DEVICE)
    per_subject_metrics[subj] = metrics
    trained_models[subj] = (model, te_loader)
    print(f"{subj} test -> Combined: MSE={metrics['Combined']['MSE']:.3f} "
          f"PCORR={metrics['Combined']['PCORR']:.3f} COSSIM={metrics['Combined']['COSSIM']:.3f}  "
          f"| IED PCORR={metrics['IED']['PCORR']:.3f}  Non-IED PCORR={metrics['Non-IED']['PCORR']:.3f}")


## 10. Results table — MSE / PCORR / COSSIM for IED, Non-IED, and Combined

In [ ]:

results_table = build_results_table(per_subject_metrics, index_name="Subject")
display(results_table)


### Shuffle-control diagnostic (run this on YOUR real data)

Trains a second model, identical in every way except the iEEG targets are **randomly
shuffled** so each scEEG segment is paired with the WRONG iEEG segment on purpose, then
compares its test PCORR to your real model's.

- **Shuffled PCORR << Real PCORR**: the model is learning genuine shared structure.
- **Shuffled PCORR ~= Real PCORR**: very little real signal is being used — check the data
  preparation (verify `X_eeg[k]`/`X_ieeg[k]` really are the same segment/subject/label).

Uses a reduced epoch budget (half of the main run) since this is a diagnostic, not the model
you'll keep.

In [ ]:

example_subj = unique_subjects[0]
d = subject_data[example_subj]
Xtr_sc_n, Xtr_ie_n, ytr = d["Xtr_sc"], d["Xtr_ie"], d["ytr"]
val_loader = DataLoader(SegSet(d["Xva_sc"], d["Xva_ie"], d["yva"]), batch_size=16, shuffle=False)
test_loader = DataLoader(SegSet(d["Xte_sc"], d["Xte_ie"], d["yte"]), batch_size=16, shuffle=False)
test_metrics = per_subject_metrics[example_subj]
print(f"Running shuffle-control diagnostic on {example_subj}...")

rng_shuffle = np.random.RandomState(SEED)
shuffle_idx = rng_shuffle.permutation(len(Xtr_ie_n))
Xtr_ie_shuffled = Xtr_ie_n[shuffle_idx]

tr_uniform_shuf = DataLoader(SegSet(Xtr_sc_n, Xtr_ie_shuffled, ytr), batch_size=BATCH_SIZE, shuffle=True)
sampler_shuf = make_balanced_sampler(ytr)
tr_weighted_shuf = DataLoader(SegSet(Xtr_sc_n, Xtr_ie_shuffled, ytr), batch_size=BATCH_SIZE, sampler=sampler_shuf)

model_shuf = TCNUNet(in_ch=M, out_ch=Mb, L=L, base_ch=BASE_CH)
model_shuf = fit_model(model_shuf, tr_uniform_shuf, tr_weighted_shuf, val_loader, device=DEVICE,
                        stage1_epochs=max(10, STAGE1_EPOCHS // 2), stage2_epochs=max(20, STAGE2_EPOCHS // 2),
                        patience=max(8, PATIENCE // 2), lr=LR, lam1=LAM1, lam2=LAM2, lam3=LAM3, lam4=LAM4,
                        lam_margin=LAM_MARGIN, target_margin=MARGIN_TARGET, verbose=False)

shuffled_metrics = score_mapping(model_shuf, test_loader, device=DEVICE)
print("Shuffled-pairing control -> Combined: "
      f"MSE={shuffled_metrics['Combined']['MSE']:.3f}  PCORR={shuffled_metrics['Combined']['PCORR']:.3f}  "
      f"COSSIM={shuffled_metrics['Combined']['COSSIM']:.3f}")
print("Real-pairing model (from above) -> Combined: "
      f"MSE={test_metrics['Combined']['MSE']:.3f}  PCORR={test_metrics['Combined']['PCORR']:.3f}  "
      f"COSSIM={test_metrics['Combined']['COSSIM']:.3f}")
if test_metrics['Combined']['PCORR'] - shuffled_metrics['Combined']['PCORR'] > 0.15:
    print("\n-> Real pairing clearly beats shuffled pairing: the model is using genuine signal.")
else:
    print("\n-> Real pairing is NOT clearly better than shuffled pairing -- investigate the "
          "data preparation before tuning the model further.")


## 12. Save results and models

In [ ]:

results_table.to_csv(os.path.join(".", "tcn_unet_subject_dependent_results.csv"))
torch.save({subj: m[0].state_dict() for subj, m in trained_models.items()},
           os.path.join(".", "tcn_unet_subject_dependent_models.pt"))
print("Saved: tcn_unet_subject_dependent_results.csv, tcn_unet_subject_dependent_models.pt")


## 13. Sanity check: real vs. estimated iEEG for one subject

In [ ]:

def plot_real_vs_est(model, loader, ch=0, n=3, title=""):
    model.eval()
    sc, ie, lab = next(iter(loader))
    with torch.no_grad():
        y_est = model(sc.to(DEVICE)).cpu().numpy()
    ie_np = ie.numpy()
    fig, axes = plt.subplots(1, n, figsize=(4*n, 3))
    for i in range(n):
        axes[i].plot(ie_np[i, :, ch], label="Real iEEG", color="black")
        axes[i].plot(y_est[i, :, ch], label="Estimated iEEG", color="crimson", alpha=0.8)
        axes[i].set_title(f"label={int(lab[i].item())}")
        axes[i].legend(fontsize=7)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

example_subj = unique_subjects[0]
model_ex, te_loader_ex = trained_models[example_subj]
plot_real_vs_est(model_ex, te_loader_ex, ch=0, title=f"{example_subj} - test set, iEEG channel {fo_names[0]}")
